# 02 · Tumor detection model (EfficientNet-B0 on 2D axial slices)

The model looks at one axial MRI slice (all 4 modalities as 4 input channels) and answers: **is there tumor on this slice?**
Labels come from the expert masks: a slice is "tumor" if it contains at least one tumor pixel.

**Runtime:** GPU (Runtime → Change runtime type → **T4 GPU**). **Needs:** notebook 01 finished (same `QUICK_RUN` setting).

**Resuming:** the full training state is saved to Drive after every epoch. If Colab disconnects, reopen and *Run all*:
training continues from the last finished epoch. Keep `RUN_NAME` the same to resume; change it to start a new, separate run.

The settings below are the final run. `QUICK_RUN = True` trains 2 short epochs on the quick data instead (smoke test).

Everything is saved to `runs/detection/<RUN_NAME>/` on Drive.

In [ ]:
# ======================= SETTINGS - edit these ===========================
PROJECT_DIR = '/content/drive/MyDrive/BrainTumor'
WORK_DIR    = '/content/work'
QUICK_RUN   = False             # False = real training. True = 2-epoch smoke test on the quick data

RUN_NAME    = 'effb0_v1'        # same name = resume, new name = new run
MODEL_NAME  = 'efficientnet_b0' # efficientnet_b0 | efficientnet_b2 | efficientnetv2_s | efficientnetv2_m | resnet18 | resnet50
PRETRAINED  = True              # start from ImageNet weights

EPOCHS            = 20
CASES_PER_BATCH   = 4           # patients per batch ...
SLICES_PER_CASE   = 16          # ... x slices per patient = 64 slices per batch
CASE_REPEATS      = 4           # each training patient is visited this many times per epoch (different slices)
LR                = 3e-4
WEIGHT_DECAY      = 1e-4
WARMUP_EPOCHS     = 1
EARLY_STOP_PATIENCE = 6         # stop if validation AUC has not improved for this many epochs (None = never)

MIN_BRAIN_PIXELS  = 500         # ignore slices with almost no brain (top / bottom of the head)
POS_MIN_PIXELS    = 1           # a slice is "tumor" if it has at least this many tumor pixels
THRESHOLD_METHOD  = 'f1'        # how the decision threshold is chosen on VALIDATION: 'f1' or 'youden'

AMP               = True        # mixed precision (faster on GPU)
NUM_WORKERS       = None        # None = automatic
SEED              = 42
KEEP_EVERY_EPOCH  = True        # also keep the weights of every epoch (small files)

In [ ]:
# ---- Setup: mount Google Drive, install missing packages, import the project code ----
import os
import sys

# Only used when testing outside Colab; on Colab these three lines change nothing.
PROJECT_DIR = os.environ.get('BT_PROJECT_DIR', PROJECT_DIR)
WORK_DIR = os.environ.get('BT_WORK_DIR', WORK_DIR)
QUICK_RUN = os.environ.get('BT_QUICK_RUN', '1' if QUICK_RUN else '0') == '1'

if PROJECT_DIR.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

CODE_DIR = os.path.join(PROJECT_DIR, 'code')
if not os.path.isfile(os.path.join(CODE_DIR, 'brats', '__init__.py')):
    raise FileNotFoundError(f'The "brats" folder was not found in {CODE_DIR}.\n'
                            'Upload everything from the "Code" folder to that Drive folder first (see README).')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from brats.notebook import install_missing
install_missing({'nibabel': 'nibabel', 'timm': 'timm', 'sklearn': 'scikit-learn'})

import copy
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from brats import config, dataset, detection, env, fileio, metrics, plots, training

if QUICK_RUN:
    EPOCHS, CASE_REPEATS, SLICES_PER_CASE, EARLY_STOP_PATIENCE = 2, 1, 8, None
paths = env.ProjectPaths(PROJECT_DIR, WORK_DIR, quick=QUICK_RUN).make()
env.print_header('QUICK RUN - smoke test only, the numbers mean nothing' if QUICK_RUN
                 else f'TRAINING RUN "{RUN_NAME}"')
device = env.get_device()
env.require_gpu(device, QUICK_RUN)
NUM_WORKERS = NUM_WORKERS if NUM_WORKERS is not None else env.default_num_workers()
env.set_seed(SEED)
print(f'device={device}  workers={NUM_WORKERS}  torch={torch.__version__}')

## 1 · Copy the preprocessed data to the local disk
Reading from the local disk is much faster than from Drive. This takes a few minutes the first time in a session.

In [ ]:
cases_dir = dataset.stage_to_local_disk(paths)
cases, slices, split = dataset.load_metadata(paths)
case_ids = dataset.split_case_ids(split)
dataset.verify_staged(cases_dir, case_ids['train'] + case_ids['val'] + case_ids['test'])
slice_tables = {name: detection.detection_slice_table(slices, ids, MIN_BRAIN_PIXELS, POS_MIN_PIXELS)
                for name, ids in case_ids.items()}
overview = pd.DataFrame([{'split': name, 'patients': len(case_ids[name]), 'slices': len(table),
                          'tumor_slices': int(table['label'].sum()),
                          'tumor_pct': 100 * table['label'].mean() if len(table) else float('nan')}
                         for name, table in slice_tables.items()])
display(overview)

## 2 · Run folder, settings and environment (all saved to Drive)

In [ ]:
run_dir = paths.run_dir('detection', RUN_NAME)
config_record = dict(
    task='detection', run_name=RUN_NAME, quick_run=QUICK_RUN, model_name=MODEL_NAME,
    timm_name=detection.DETECTION_MODELS.get(MODEL_NAME, MODEL_NAME), pretrained=PRETRAINED,
    epochs=EPOCHS, cases_per_batch=CASES_PER_BATCH, slices_per_case=SLICES_PER_CASE, case_repeats=CASE_REPEATS,
    lr=LR, weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, early_stop_patience=EARLY_STOP_PATIENCE,
    min_brain_pixels=MIN_BRAIN_PIXELS, pos_min_pixels=POS_MIN_PIXELS, threshold_method=THRESHOLD_METHOD, amp=AMP,
    seed=SEED, in_channels=list(config.MODALITIES), canvas=config.DETECTION_CANVAS, norm_clip=config.NORM_CLIP,
    n_cases=overview.set_index('split')['patients'].to_dict(), n_slices=overview.set_index('split')['slices'].to_dict(),
    data_dir=str(paths.data))
config_path = run_dir / 'config.json'
if config_path.is_file():
    previous = fileio.load_json(config_path)
    if previous.get('model_name') != MODEL_NAME:
        raise RuntimeError(f'Run "{RUN_NAME}" was trained with {previous.get("model_name")}. '
                           f'Use a new RUN_NAME for {MODEL_NAME}.')
    fileio.save_json(config_record, run_dir / 'logs' / f'config_resume_{time.strftime("%Y%m%d_%H%M%S")}.json')
else:
    fileio.save_json(config_record, config_path)
env.save_environment(run_dir / 'logs')
env.snapshot_code(run_dir, paths.code, '02_detection_training.ipynb')
for name, table in slice_tables.items():
    fileio.save_csv(table, run_dir / 'logs' / f'slice_table_{name}.csv')
print('Run folder:', run_dir)

## 3 · Data loaders and a look at one training batch

In [ ]:
train_set = detection.DetectionDataset(slice_tables['train'], cases_dir, SLICES_PER_CASE, CASE_REPEATS, augment=True)
train_loader = training.make_loader(train_set, CASES_PER_BATCH, True, NUM_WORKERS, device, detection.collate_patients,
                                    drop_last=len(train_set) >= 2 * CASES_PER_BATCH)
eval_loaders = {name: training.make_loader(detection.DetectionDataset(slice_tables[name], cases_dir), 1, False,
                                           NUM_WORKERS, device, detection.collate_patients)
                for name in ('train', 'val', 'test')}
print(f'{len(train_loader)} training batches per epoch, {CASES_PER_BATCH * SLICES_PER_CASE} slices each')

inputs, labels, info = next(iter(train_loader))
print('batch:', tuple(inputs.shape), inputs.dtype, '| labels:', labels[:16].int().tolist(),
      f'| value range {inputs.float().min():.2f} .. {inputs.float().max():.2f}')
assert inputs.shape[1:] == (4, config.DETECTION_CANVAS, config.DETECTION_CANVAS) and torch.isfinite(inputs.float()).all()
n_shown = min(16, len(inputs))
fig = plots.image_grid(
    [plots.view2d(inputs[i, 0].float().numpy()) for i in range(n_shown)],
    titles=[f'{"tumor" if labels[i] > 0 else "no tumor"}\n{info["case_id"][i][-5:]} z{info["z_orig"][i]}'
            for i in range(n_shown)],
    ncols=8, cell=1.7, suptitle='One training batch (FLAIR channel, after augmentation)')
plots.save_fig(fig, run_dir / 'figures', '01_training_batch')
del inputs, labels, info

## 4 · Model, optimizer, and resume from the last checkpoint (if any)

In [ ]:
model, imagenet_loaded = detection.build_detection_model(MODEL_NAME, in_chans=4, pretrained=PRETRAINED)
model = model.to(device)
n_params = training.count_parameters(model)
tumor_fraction = float(slice_tables['train']['label'].mean()) if len(slice_tables['train']) else 0.5
pos_weight = (1 - tumor_fraction) / tumor_fraction if 0 < tumor_fraction < 1 else 1.0   # balances the two classes
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = max(1, len(train_loader))
scheduler = training.make_scheduler(optimizer, EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch)
scaler = training.make_grad_scaler(device, AMP)
print(f'{MODEL_NAME}: {n_params / 1e6:.2f} M parameters, ImageNet weights loaded: {imagenet_loaded}, '
      f'tumor slices in training: {100 * tumor_fraction:.1f}% (pos_weight {pos_weight:.2f})')

run = training.ResumableTraining(run_dir, model, optimizer, scheduler, scaler, device, model_name=MODEL_NAME,
                                 config=config_record, metric_name='val_auc', epochs=EPOCHS,
                                 patience=EARLY_STOP_PATIENCE, keep_every_epoch=KEEP_EVERY_EPOCH)

## 5 · Training
After every epoch: validation on all validation slices, `history.csv` updated, `last.pt` (full state) saved,
`best.pt` updated when validation AUC improves. Interrupting is safe.

In [ ]:
for epoch in run.remaining_epochs():
    start = time.time()
    train_stats = training.train_one_epoch(model, train_loader, optimizer, scheduler, scaler, loss_fn, device, AMP,
                                           max_grad_norm=5.0, track_accuracy=True)
    val = detection.predict_loader(model, eval_loaders['val'], device, AMP, desc='val')
    val_metrics = metrics.binary_metrics(val['label'], val['prob'], 0.5)
    score = val_metrics['auc'] if np.isfinite(val_metrics['auc']) else val_metrics['accuracy']
    row = {**train_stats, 'val_loss': metrics.weighted_bce(val['label'], val['prob'], pos_weight),
           'val_auc': val_metrics['auc'], 'val_ap': val_metrics['average_precision'],
           'val_acc': val_metrics['accuracy'], 'val_f1': val_metrics['f1'],
           'val_sensitivity': val_metrics['sensitivity'], 'val_specificity': val_metrics['specificity'],
           'val_precision': val_metrics['precision'], 'lr': optimizer.param_groups[0]['lr'],
           'epoch_seconds': time.time() - start}
    improved = run.end_epoch(epoch, row, metric=score)
    print(f'epoch {epoch:3d}/{EPOCHS} | train loss {train_stats["train_loss"]:.4f} acc {train_stats["train_acc"]:.3f} | '
          f'val loss {row["val_loss"]:.4f} AUC {val_metrics["auc"]:.4f} F1 {val_metrics["f1"]:.3f} | '
          f'{row["epoch_seconds"] / 60:.1f} min{"  * best" if improved else ""}')
print(f'Training finished. Best validation AUC {run.best_metric:.4f} at epoch {run.best_epoch}.')

## 6 · Final predictions with the best model
The decision threshold is chosen on the **validation** set; the **test** set is only used for the final numbers.

In [ ]:
model.load_state_dict(training.load_model_weights(run.best_path, device)['model'])
model.eval()
predictions = {}
for name in ('train', 'val', 'test'):
    predictions[name] = detection.predict_loader(model, eval_loaders[name], device, AMP, desc=name)
    predictions[name].insert(0, 'split', name)
threshold = metrics.choose_threshold(predictions['val']['label'], predictions['val']['prob'], THRESHOLD_METHOD)
for name, table in predictions.items():
    table['pred'] = (table['prob'] >= threshold).astype(int)
    fileio.save_csv(table, run_dir / 'predictions' / f'{name}_slices.csv')
split_metrics = {name: metrics.binary_metrics(t['label'], t['prob'], threshold) for name, t in predictions.items()}
split_metrics_at_05 = {name: metrics.binary_metrics(t['label'], t['prob'], 0.5) for name, t in predictions.items()}
patients = {name: detection.patient_detection_summary(t, threshold) for name, t in predictions.items()}
for name, table in patients.items():
    fileio.save_csv(table, run_dir / 'predictions' / f'{name}_patients.csv')
metrics_table = pd.DataFrame(split_metrics).T
fileio.save_csv(metrics_table.reset_index().rename(columns={'index': 'split'}),
                run_dir / 'predictions' / 'metrics_by_split.csv')
display(metrics_table[['n', 'accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'auc',
                       'average_precision']].round(4))
print(f'Decision threshold (chosen on validation by {THRESHOLD_METHOD}): {threshold:.3f}')

In [ ]:
# Inference speed (GPU and CPU) and exports for the offline app
speed = {}
dummy = torch.zeros(64, 4, config.DETECTION_CANVAS, config.DETECTION_CANVAS, device=device)
with torch.no_grad():
    for _ in range(2):
        model(dummy[:8])                                   # warm-up
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    speed[f'{device.type}_ms_per_slice'] = 1000 * (time.time() - start) / 64
    cpu_model = copy.deepcopy(model).cpu().eval()
    start = time.time()
    cpu_model(torch.zeros(8, 4, config.DETECTION_CANVAS, config.DETECTION_CANVAS))
    speed['cpu_ms_per_slice'] = 1000 * (time.time() - start) / 8
print('Inference time:', {key: round(value, 2) for key, value in speed.items()})
try:
    traced = torch.jit.trace(cpu_model, torch.zeros(1, 4, config.DETECTION_CANVAS, config.DETECTION_CANVAS))
    traced.save(str(run_dir / 'export' / 'detector_torchscript.pt'))
    print('TorchScript export saved.')
except Exception as error:      # the export is a bonus; never fail the run because of it
    print('TorchScript export skipped:', error)
del cpu_model, dummy

summary = {
    'run_name': RUN_NAME, 'quick_run': QUICK_RUN, 'model_name': MODEL_NAME, 'parameters': n_params,
    'pretrained_loaded': imagenet_loaded, 'best_epoch': run.best_epoch, 'best_val_auc': run.best_metric,
    'epochs_trained': len(run.history), 'training_hours': run.training_hours,
    'threshold': threshold, 'threshold_method': THRESHOLD_METHOD,
    'metrics': split_metrics, 'metrics_at_0.5': split_metrics_at_05,
    'patients_detected': {name: f'{int(t["tumor_present_pred"].sum())}/{len(t)}' for name, t in patients.items()},
    'mean_patient_slice_iou': {name: float(t['slice_set_iou'].mean()) for name, t in patients.items()},
    'inference': speed,
}
fileio.save_json(summary, run_dir / 'summary.json')
# Everything the offline app needs to use this model:
fileio.save_json({'task': 'detection', 'weights': 'checkpoints/best.pt', 'model_name': MODEL_NAME,
                  'timm_name': config_record['timm_name'], 'in_channels': list(config.MODALITIES),
                  'canvas': config.DETECTION_CANVAS,
                  'normalization': f'z-score per modality over non-zero voxels, clip +-{config.NORM_CLIP}',
                  'min_brain_pixels': MIN_BRAIN_PIXELS, 'threshold': threshold},
                 run_dir / 'model_card.json')
print('Saved summary.json and model_card.json')

## 7 · Graphs
All saved in `runs/detection/<RUN_NAME>/figures/` (PNG + PDF + CSV).

In [ ]:
FIG = run_dir / 'figures'
history = pd.DataFrame(run.history)
fig = plots.training_curves(run.history, [
    ('Loss', [('train_loss', 'train'), ('val_loss', 'validation')], 'loss'),
    ('Accuracy', [('train_acc', 'train'), ('val_acc', 'validation')], 'accuracy'),
    ('Validation AUC / AP', [('val_auc', 'AUC'), ('val_ap', 'average precision')], 'score'),
    ('Validation F1 (threshold 0.5)', [('val_f1', 'F1')], 'F1'),
    ('Validation sensitivity / specificity', [('val_sensitivity', 'sensitivity'), ('val_specificity', 'specificity')], 'score'),
    ('Validation precision', [('val_precision', 'precision')], 'precision'),
], title=f'Detection training - {MODEL_NAME}')
plots.save_fig(fig, FIG, '02_training_curves', data=history)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(history['epoch'], history['lr'], marker='o', color=plots.PALETTE[0])
axes[0].set_title('Learning rate (end of epoch)')
axes[1].bar(history['epoch'], history['epoch_seconds'] / 60, color=plots.PALETTE[1])
axes[1].set_title('Time per epoch')
axes[1].set_ylabel('minutes')
axes[2].bar(history['epoch'], history['gpu_mem_gb'], color=plots.PALETTE[2])
axes[2].set_title('Peak GPU memory')
axes[2].set_ylabel('GB')
for ax in axes:
    ax.set_xlabel('epoch')
fig.tight_layout()
plots.save_fig(fig, FIG, '03_lr_time_memory', data=history[['epoch', 'lr', 'epoch_seconds', 'gpu_mem_gb']])

In [ ]:
for name in ('val', 'test'):
    m = split_metrics[name]
    counts = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
    plots.save_fig(plots.confusion_matrix_fig(counts, ['no tumor', 'tumor'], f'Detection confusion matrix - {name}'),
                   FIG, f'04_confusion_{name}', data=pd.DataFrame(counts, columns=['pred_no_tumor', 'pred_tumor']))
    plots.save_fig(plots.confusion_matrix_fig(counts, ['no tumor', 'tumor'], f'Detection confusion matrix - {name}',
                                              normalize=True), FIG, f'05_confusion_{name}_normalized')

curves = {name: (predictions[name]['label'].values, predictions[name]['prob'].values) for name in ('train', 'val', 'test')}
fig, table = plots.roc_fig(curves, 'ROC curve (slice level)')
plots.save_fig(fig, FIG, '06_roc', data=table)
fig, table = plots.pr_fig(curves, 'Precision-recall curve (slice level)')
plots.save_fig(fig, FIG, '07_precision_recall', data=table)
fig, table = plots.threshold_sweep_fig(predictions['val']['label'].values, predictions['val']['prob'].values, threshold,
                                       'Validation metrics vs decision threshold')
plots.save_fig(fig, FIG, '08_threshold_sweep', data=table)
test_labels, test_probs = predictions['test']['label'].values, predictions['test']['prob'].values
calibration, ece = metrics.calibration_table(test_labels, test_probs)
plots.save_fig(plots.calibration_fig(calibration, ece, test_labels, test_probs,
                                     'Are the confidence scores trustworthy? (test)'),
               FIG, '09_calibration', data=calibration)

In [ ]:
# Where does the model make mistakes? (test set)
test = predictions['test'].copy()
tumor_slices = test[test['label'] == 1].copy()
size_edges = [1, 50, 200, 500, 1000, 2000, 5000, 1e9]
size_names = ['1-49', '50-199', '200-499', '500-999', '1000-1999', '2000-4999', '5000+']
tumor_slices['size_bin'] = pd.cut(tumor_slices['wt_px'], bins=size_edges, labels=size_names, right=False)
by_size = (tumor_slices.groupby('size_bin', observed=False)
           .agg(n=('pred', 'size'), sensitivity=('pred', 'mean')).reset_index())
fig, ax = plt.subplots(figsize=(8, 3.8))
plots.bar_panel(ax, by_size['size_bin'].astype(str).tolist(), (100 * by_size['sensitivity'].fillna(0)).tolist(),
                title='Sensitivity by tumor size on the slice (test)', ylabel='% of tumor slices detected', fmt='{:.0f}%')
ax.set_xlabel('tumor pixels on the slice')
fig.tight_layout()
plots.save_fig(fig, FIG, '10_sensitivity_by_tumor_size', data=by_size)

positions = test.merge(cases[['case_id', 'bbox_z0', 'bbox_z1']], on='case_id')
positions['rel_pos'] = (positions['z_orig'] - positions['bbox_z0']) / (positions['bbox_z1'] - positions['bbox_z0']).clip(lower=1)
positions['pos_bin'] = pd.cut(positions['rel_pos'], np.linspace(0, 1, 11), include_lowest=True)
positions['error'] = (positions['pred'] != positions['label']).astype(float)
by_position = (positions.groupby('pos_bin', observed=False)
               .agg(n=('error', 'size'), error_rate=('error', 'mean')).reset_index())
fig, ax = plt.subplots(figsize=(8, 3.8))
plots.bar_panel(ax, [f'{i * 10}-{i * 10 + 10}%' for i in range(len(by_position))],
                (100 * by_position['error_rate'].fillna(0)).tolist(),
                title='Error rate by slice position in the brain (test)', ylabel='% of slices wrong', fmt='{:.1f}')
ax.set_xlabel('position from the bottom to the top of the brain')
ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
plots.save_fig(fig, FIG, '11_errors_by_slice_position', data=by_position.astype({'pos_bin': str}))

In [ ]:
# Example slices: correct and wrong decisions (test)
def example_grid(rows: pd.DataFrame, title: str, name: str, n: int = 12, seed: int = 0) -> None:
    rows = rows.sample(n=min(n, len(rows)), random_state=seed) if len(rows) else rows
    images, titles, outlines = [], [], []
    for case_id, group in rows.groupby('case_id'):
        case = dataset.load_case(cases_dir, case_id)
        slice_inputs = detection.detection_inputs(case, group['z'].values)
        slice_masks = detection.detection_masks(case, group['z'].values)
        for i, (_, row) in enumerate(group.iterrows()):
            images.append(plots.view2d(slice_inputs[i, 0]))
            outlines.append([(plots.view2d(slice_masks[i]), plots.PALETTE[2])])
            titles.append(f'p={row["prob"]:.2f}  {case_id[-5:]} z{row["z_orig"]}')
    fig = plots.image_grid(images, titles, contours=outlines, ncols=6,
                           suptitle=title + ' (green outline = expert tumor)')
    plots.save_fig(fig, FIG, name, data=rows)

example_grid(test[(test['label'] == 1) & (test['pred'] == 1)], 'True positives', '12_examples_true_positive')
example_grid(test[(test['label'] == 0) & (test['pred'] == 1)], 'False positives', '13_examples_false_positive')
example_grid(test[(test['label'] == 1) & (test['pred'] == 0)], 'False negatives (missed tumor)',
             '14_examples_false_negative')
example_grid(test[(test['label'] == 0) & (test['pred'] == 0)], 'True negatives', '15_examples_true_negative')

In [ ]:
# Per-patient view: tumor probability along the head vs the true tumor slices
shown_ids = sorted(test['case_id'].unique())[:6]
fig, axes = plt.subplots(len(shown_ids), 1, figsize=(9, 1.6 * len(shown_ids) + 0.6), squeeze=False, sharex=True)
for ax, case_id in zip(axes[:, 0], shown_ids):
    patient = test[test['case_id'] == case_id].sort_values('z_orig')
    for z in patient.loc[patient['label'] == 1, 'z_orig']:
        ax.axvspan(z - 0.5, z + 0.5, color=plots.PALETTE[1], alpha=0.18, linewidth=0)
    ax.plot(patient['z_orig'], patient['prob'], color=plots.PALETTE[0])
    ax.axhline(threshold, color=plots.INK2, linestyle='--', linewidth=0.8)
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel(case_id[-5:], rotation=0, ha='right', va='center')
axes[-1, 0].set_xlabel('axial slice index (orange band = expert tumor slices, dashed = threshold)')
fig.suptitle('Tumor probability per slice for test patients', y=1.0)
fig.tight_layout()
plots.save_fig(fig, FIG, '16_patient_slice_profiles', data=test[test['case_id'].isin(shown_ids)])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
plots.hist_panel(axes[0], patients['test']['slice_accuracy'], bins=20, title='Slice accuracy per test patient',
                 xlabel='accuracy')
plots.hist_panel(axes[1], patients['test']['slice_set_iou'], bins=20, color=plots.PALETTE[1],
                 title='Overlap of predicted vs true tumor slices', xlabel='IoU of slice sets')
fig.tight_layout()
plots.save_fig(fig, FIG, '17_patient_level_scores', data=patients['test'])

In [ ]:
# Compare all detection runs that have finished so far
comparison = detection.compare_detection_runs(paths.runs / 'detection', QUICK_RUN)
display(comparison)
fileio.save_csv(comparison, paths.runs / 'detection' / ('comparison_quick.csv' if QUICK_RUN else 'comparison.csv'))
fig = plots.run_comparison_fig(comparison, ['test_auc', 'test_f1', 'test_sensitivity', 'test_specificity', 'params_M',
                                            'cpu_ms_per_slice'], title='Detection runs compared (test set)')
plots.save_fig(fig, FIG, '18_run_comparison', data=comparison)

## Done
Saved in `runs/detection/<RUN_NAME>/`: `checkpoints/best.pt` (for the app), `last.pt`, per-epoch weights, `history.csv`,
`predictions/` (every slice of train/val/test), `summary.json`, `model_card.json`, `figures/`, `logs/`, `code_snapshot/`, `export/`.

In [ ]:
env.flush_drive()